# 04b - Modelo 1: DetecciÃ³n Binaria de AnomalÃ­as con LightGBM
 
 Variante paralela del cuaderno 04. Mantiene el mismo split temporal, features, imputaciÃ³n e indicadores de `NaN`, pero sustituye el clasificador base por `LGBMClassifier`.
 
 **Entrada:** `data/interim/03_datos_features_full_v3.parquet`  
 **Salida:** artefactos con sufijo `lightgbm` en `data/models/` y `data/interim/04b_modelo1_predicciones_lightgbm_full_v3.parquet`


In [ ]:
# â”€â”€â”€ Intel Extension for Scikit-learn â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# â”€â”€â”€ LibrerÃ­as estÃ¡ndar â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# â”€â”€â”€ Scikit-learn (ya parcheado si Intel Extension estÃ¡ disponible) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# â”€â”€â”€ ConfiguraciÃ³n del proyecto â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import sys
PROJECT_DIR = os.getcwd()
if not os.path.exists(os.path.join(PROJECT_DIR, 'config.py')):
    raise FileNotFoundError(
        f"No se encuentra config.py en el directorio actual: {PROJECT_DIR}. "
        "Abre y ejecuta el notebook desde la carpeta del proyecto."
    )
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
import config as project_config
from config import *
print(f"Config cargado desde: {project_config.__file__}")

# Rutas aisladas experimento FULL_V3
EXPERIMENTO_FULL = 'full_v3'
PARQUET_03 = os.path.join(DATA_INTERIM, '03_datos_features_full_v3.parquet')
PARQUET_04_LGBM = os.path.join(DATA_INTERIM, '04b_modelo1_predicciones_lightgbm_full_v3.parquet')
MODELO_1_LGBM_PATH = os.path.join(DATA_MODELS, 'modelo_1_detector_lightgbm_full_v3.joblib')
IMPUTER_M1_LGBM_PATH = os.path.join(DATA_MODELS, 'imputer_modelo_1_lightgbm_full_v3.joblib')
FEATURES_M1_LGBM_PATH = os.path.join(DATA_MODELS, 'features_modelo_1_lightgbm_full_v3.joblib')
HISTORIAL_04B_LGBM_PATH = os.path.join(DATA_RESULTS, 'historial_ejecuciones_04b_modelo_deteccion_lightgbm_full_v3.txt')
BASELINES_CTX_LGBM_PATH = os.path.join(DATA_MODELS, 'baselines_ctx_lightgbm_full_v3.joblib')
print('Experimento:', EXPERIMENTO_FULL)
print('Features:', PARQUET_03)
print('Predicciones M1:', PARQUET_04_LGBM)

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA â€” algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos y preparar X, y

In [ ]:
# Cargar el dataset con features calculadas
df_trabajo = pd.read_parquet(PARQUET_03)
print(f"Dataset cargado: {df_trabajo.shape}")

if COLUMNA_USAR_EN_TRAIN in df_trabajo.columns:
    resumen_uso = df_trabajo[COLUMNA_USAR_EN_TRAIN].value_counts(dropna=False)
    print(f"Distribucion de '{COLUMNA_USAR_EN_TRAIN}':")
    print(resumen_uso)

    filas_totales = len(df_trabajo)
    mask_modelado = df_trabajo[COLUMNA_USAR_EN_TRAIN].fillna(False).astype(bool)
    df_trabajo = df_trabajo.loc[mask_modelado].reset_index(drop=True)
    cobertura = len(df_trabajo) / filas_totales * 100 if filas_totales else 0.0
    print(f"Filas usadas para modelado y metricas: {len(df_trabajo):,} / {filas_totales:,} ({cobertura:.2f}%)")
else:
    print(f"Columna '{COLUMNA_USAR_EN_TRAIN}' no disponible: PARQUET_03 ya viene filtrado por train; se usan todas sus filas.")

# Separar features (X) y etiqueta binaria (y)
excluir = COLUMNAS_EXCLUIR_FEATURES
columnas_features = [c for c in df_trabajo.columns if c not in excluir]
X = df_trabajo[columnas_features].select_dtypes(include='number').copy()
y = df_trabajo['etiqueta_deteccion'].copy()

print(f"Features: {X.shape[1]} columnas")
print('Distribucion de clases:')
print(y.value_counts())


## 1. TimeSeriesSplit â€” k=4 folds (v3 â€” M6)
Dividimos en 4 partes ordenadas temporalmente. Esto evita data leakage: el modelo nunca ve datos futuros durante el entrenamiento.
Calculamos accuracy, F1 y ROC-AUC por fold para ver estabilidad temporal.

In [ ]:
# v3 â€” M6: TimeSeriesSplit con k=4 folds â€” sin data leakage, evaluaciÃ³n en todas las estaciones
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

if 'X' in locals() and 'y' in locals():
    tscv = TimeSeriesSplit(n_splits=TSCV_N_SPLITS)

    resultados_folds = []
    fold_splits = []  # guardamos los Ã­ndices para reusar el Ãºltimo fold en correcciÃ³n

    print('TimeSeriesSplit k=4 â€” distribuciÃ³n de folds:')
    print('=' * 60)

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
        X_train_f = X.iloc[train_idx]
        X_test_f  = X.iloc[test_idx]
        y_train_f = y.iloc[train_idx]
        y_test_f  = y.iloc[test_idx]

        # Fechas aproximadas de cada fold (si Fecha estÃ¡ disponible)
        if 'Fecha' in df_trabajo.columns:
            fecha_train_ini = df_trabajo['Fecha'].iloc[train_idx[0]]
            fecha_train_fin = df_trabajo['Fecha'].iloc[train_idx[-1]]
            fecha_test_ini  = df_trabajo['Fecha'].iloc[test_idx[0]]
            fecha_test_fin  = df_trabajo['Fecha'].iloc[test_idx[-1]]
            print(f'Fold {fold}:')
            print(f'  Train: {fecha_train_ini:%Y-%m-%d} â†’ {fecha_train_fin:%Y-%m-%d} ({len(train_idx):,} filas)')
            print(f'  Test:  {fecha_test_ini:%Y-%m-%d} â†’ {fecha_test_fin:%Y-%m-%d} ({len(test_idx):,} filas)')
        else:
            print(f'Fold {fold}: train={len(train_idx):,} filas  test={len(test_idx):,} filas')

        fold_splits.append((train_idx, test_idx))

    print('=' * 60)
    print()
    print('Los 4 modelos se entrenan en la celda siguiente.')
    print('El Fold 4 (train mÃ¡s largo) se usa para correcciÃ³n y comparaciÃ³n final.')

    # Exponer el fold 4 como split principal para el resto del notebook
    train_idx_final, test_idx_final = fold_splits[-1]
    X_train = X.iloc[train_idx_final]
    X_test  = X.iloc[test_idx_final]
    y_train = y.iloc[train_idx_final]
    y_test  = y.iloc[test_idx_final]

else:
    print('Error: X o y no definidos.')


## 2. Entrenar modelo final en el Ãºltimo fold
Usamos el split del fold final (mÃ¡s reciente) para tener el modelo entrenado con el mayor volumen de datos posible.

In [ ]:
# â”€â”€ Helper: baselines contextuales por hora â€” fold-aware â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Computa std esperada por hora del train y genera {col}_rstd_vs_expected:
# "Â¿estÃ¡ el sensor MÃS quieto de lo normal para esta hora?"
#   SA:             rstd_5m â‰ˆ 0  â†’ ratio â‰ˆ 0   â†’ anomalÃ­a
#   Normal activo:  rstd_5m â‰ˆ std_hora â†’ ratio â‰ˆ 1   â†’ normal
#   Primavera/mayo: rstd_5m alto â†’ ratio â‰¥ 1   â†’ normal, no genera FP
#
# NO usa dev ni zscore (dependen del valor absoluto â†’ FP por drift estacional).
# Usa HoraÃ—Mes cuando el mes estÃ¡ disponible; meses fuera del train â†’ fillna(1) â†’ ratio=1 (neutro).

def _add_ctx_features(X_tr_ref, X_apply, sensores):
    """Devuelve (X_apply_aumentado, lookup_stats_df)."""
    cols_grp  = [c for c in ['Hora', 'Mes'] if c in X_tr_ref.columns]
    cols_sens = [c for c in sensores
                 if c in X_tr_ref.columns and c in X_apply.columns]
    if not cols_grp or not cols_sens:
        return X_apply.copy(), None

    # Solo std por hora â€” la media no se usa (causa FP estacionales)
    agg = X_tr_ref.groupby(cols_grp)[cols_sens].std()
    agg.columns = [f'{c}_std_hora_ctx' for c in agg.columns]
    agg = agg.reset_index()

    result = X_apply.copy()
    merged = result[cols_grp].merge(agg, on=cols_grp, how='left')
    merged.index = result.index

    for col in cols_sens:
        std_c  = f'{col}_std_hora_ctx'
        rstd_c = f'{col}_rstd_5m'
        if std_c in merged.columns and rstd_c in result.columns:
            result[f'{col}_rstd_vs_expected'] = (
                result[rstd_c] / (merged[std_c].fillna(1).values + 1e-6)
            )
    return result, agg

sensores_ctx = [c for c in FE_SENSORES if c in X.columns]
print(f"Helper _add_ctx_features listo â€” genera {{col}}_rstd_vs_expected para {len(sensores_ctx)} sensores")


In [ ]:
# v3 â€” M6: Entrenar un modelo por fold y calcular mÃ©tricas por estaciÃ³n
# v5 â€” M7: Baselines contextuales (mean/std por HoraÃ—Mes) computados fold-aware
try:
    from lightgbm import LGBMClassifier
except ImportError as e:
    raise ImportError('LightGBM no estÃ¡ instalado. Instala con: pip install lightgbm') from e
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score

if 'fold_splits' in locals() and 'X' in locals() and 'y' in locals():
    metricas_folds  = []
    stats_ctx_final = None  # stats del fold final para aplicar al split/inferencia

    for fold, (train_idx, test_idx) in enumerate(fold_splits, start=1):
        X_tr_raw = X.iloc[train_idx].copy()
        X_te_raw = X.iloc[test_idx].copy()
        y_tr     = y.iloc[train_idx]
        y_te     = y.iloc[test_idx]

        # â”€â”€ M7: Baselines ctx â€” stats solo del train de este fold â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
        X_tr, stats_fold = _add_ctx_features(X_tr_raw, X_tr_raw, sensores_ctx)
        X_te, _          = _add_ctx_features(X_tr_raw, X_te_raw, sensores_ctx)

        # Imputar NaN (fit solo en train)
        imp = SimpleImputer(strategy='median', add_indicator=True)
        X_tr_imp = imp.fit_transform(X_tr)
        X_te_imp = imp.transform(X_te)

        # Entrenar LightGBM
        rf = LGBMClassifier(**MODELO_1_LGBM_PARAMS)
        rf.fit(X_tr_imp, y_tr)
        y_pred = rf.predict(X_te_imp)

        acc = accuracy_score(y_te, y_pred)
        f1  = f1_score(y_te, y_pred, average='weighted')

        if 'Fecha' in df_trabajo.columns:
            fecha_test_ini = df_trabajo['Fecha'].iloc[test_idx[0]]
            fecha_test_fin = df_trabajo['Fecha'].iloc[test_idx[-1]]
            periodo = f'{fecha_test_ini:%b %Y} \u2192 {fecha_test_fin:%b %Y}'
        else:
            periodo = f'filas {test_idx[0]}\u2013{test_idx[-1]}'

        metricas_folds.append({
            'Fold': fold,
            'PerÃ­odo test': periodo,
            'Accuracy': round(acc * 100, 2),
            'F1-weighted': round(f1, 4),
            'Train filas': len(train_idx),
            'Test filas': len(test_idx),
        })
        print(f'Fold {fold} ({periodo}): Accuracy={acc*100:.2f}%  F1={f1:.4f}')

        if fold == len(fold_splits):
            rf_detector     = rf
            imputer         = imp
            X_train_imputed = X_tr_imp
            X_test_imputed  = X_te_imp
            y_pred_detector = y_pred
            stats_ctx_final = stats_fold

    df_metricas = pd.DataFrame(metricas_folds)
    print()
    print('=' * 65)
    print('RESUMEN TimeSeriesSplit k=4')
    print('=' * 65)
    print(df_metricas.to_string(index=False))
    print(f"\nMedia  Accuracy: {df_metricas['Accuracy'].mean():.2f}% \u00b1 {df_metricas['Accuracy'].std():.2f}%")
    print(f"Media  F1:       {df_metricas['F1-weighted'].mean():.4f} \u00b1 {df_metricas['F1-weighted'].std():.4f}")
    print('=' * 65)
    print(f'Modelo del Fold 4 guardado. Baselines ctx: {len(sensores_ctx)} sensores \u00d7 (Hora, Mes)')

else:
    print('Error: fold_splits no definido. Ejecuta la celda anterior.')


In [ ]:
# â”€â”€ Aplicar baselines contextuales al split final (fold 4) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# X_train y X_test vienen del TimeSeriesSplit en la celda 194c543b.
# Usamos los stats del fold 4 (stats_ctx_final) para evitar recalcular
# y garantizar que la imputaciÃ³n y el modelo final usen las mismas referencias.

if 'stats_ctx_final' in locals() and stats_ctx_final is not None:
    X_train_raw = X_train.copy()   # referencia limpia para aplicar a X_test
    X_train, _  = _add_ctx_features(X_train_raw, X_train_raw, sensores_ctx)
    X_test,  _  = _add_ctx_features(X_train_raw, X_test,      sensores_ctx)
    cols_ctx = [c for c in X_train.columns if c.endswith('_rstd_vs_expected')]
    print(f"Baselines ctx aplicados: {len(cols_ctx)} features nuevas")
    print(f"  X_train: {X_train.shape}  |  X_test: {X_test.shape}")
    print(f"  Ejemplo: {cols_ctx[:4]}")
else:
    X_train_raw = X_train.copy()
    print("Warning: stats_ctx_final no disponible â€” ctx features no aÃ±adidas")

## 3. Imputar NaNs con SimpleImputer + indicadores
El imputer rellena NaNs con la mediana y aÃ±ade columnas binarias `missingindicator_X` para que el modelo sepa dÃ³nde habÃ­a NaN (el hecho de ser NaN es en sÃ­ una seÃ±al de anomalÃ­a).

In [ ]:
if 'X_train' in locals() and 'X_test' in locals():
    print("Manejo de NaNs antes de la imputaciÃ³n:")
    print(f"NaNs en X_train: {X_train.isnull().sum().sum()}")
    print(f"NaNs en X_test: {X_test.isnull().sum().sum()}")

    # Crear el imputador usando la mediana Y AÃ‘ADIENDO COLUMNAS INDICADORAS
    imputer = SimpleImputer(strategy='median', add_indicator=True)

    # Ajustar el imputador SOLO con los datos de entrenamiento
    # Nota: fit no devuelve nada, solo configura el imputer
    imputer.fit(X_train) 
    
    # Aplicar la imputaciÃ³n (transformar) a los datos de entrenamiento y prueba
    # El resultado serÃ¡ un array de NumPy
    X_train_imputed_np = imputer.transform(X_train)
    X_test_imputed_np = imputer.transform(X_test)

    # Obtener los nombres de las nuevas caracterÃ­sticas (originales + indicadoras)
    try:
        # Intenta obtener los nombres de las caracterÃ­sticas, incluyendo las indicadoras
        feature_names_imputed = imputer.get_feature_names_out(input_features=X_train.columns)
    except AttributeError:
        print("Advertencia: imputer.get_feature_names_out no disponible. Usando nombres genÃ©ricos para indicadoras si es necesario.") 
        # Si no se puede obtener, usamos un enfoque alternativo
        feature_names_imputed = list(X_train.columns)
        # Ver cuÃ¡ntas columnas indicadoras se aÃ±adieron
        num_original_features = X_train.shape[1]
        num_indicator_features = X_train_imputed_np.shape[1] - num_original_features
        if num_indicator_features > 0:
            # Obtener Ã­ndices de caracterÃ­sticas que tenÃ­an NaNs y para las cuales se creÃ³ un indicador
            indicator_indices = imputer.indicator_.features_ # Indices de las cols originales que generaron indicador
            indicator_col_names = [f"missing_{X_train.columns[i]}" for i in indicator_indices]
            feature_names_imputed.extend(indicator_col_names)
        if len(feature_names_imputed) != X_train_imputed_np.shape[1]: # ComprobaciÃ³n de seguridad
            print(f"Discrepancia en nombres de columnas: {len(feature_names_imputed)} vs {X_train_imputed_np.shape[1]}. Feature importance podrÃ­a no tener nombres correctos.")
            feature_names_imputed = None # Forzar a usar array si los nombres no cuadran


    # Convertir de nuevo a DataFrames de Pandas
    if feature_names_imputed is not None:
        X_train_imputed = pd.DataFrame(X_train_imputed_np, columns=feature_names_imputed, index=X_train.index)
        X_test_imputed = pd.DataFrame(X_test_imputed_np, columns=feature_names_imputed, index=X_test.index)
    else: # Si no pudimos obtener los nombres, trabajamos con arrays de NumPy (menos ideal para feature importance explÃ­cita)
        X_train_imputed = X_train_imputed_np
        X_test_imputed = X_test_imputed_np
        print("Advertencia: X_train_imputed y X_test_imputed son arrays NumPy. Nombres de caracterÃ­sticas no disponibles para la tabla de importancia.")


    print("\nManejo de NaNs despuÃ©s de la imputaciÃ³n (con indicadores):")
    # Si son DataFrames, podemos hacer isnull().sum().sum(). Si son arrays, no directamente.
    if isinstance(X_train_imputed, pd.DataFrame):
        print(f"NaNs en X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
        print(f"NaNs en X_test_imputed: {X_test_imputed.isnull().sum().sum()}")
    else: # Si son arrays NumPy, SimpleImputer deberÃ­a haber manejado todos los NaNs
        print(f"NaNs en X_train_imputed (array NumPy): {np.isnan(X_train_imputed).sum()}")
        print(f"NaNs en X_test_imputed (array NumPy): {np.isnan(X_test_imputed).sum()}")

    print(f"Forma de X_train_imputed: {X_train_imputed.shape} (puede tener mÃ¡s columnas debido a los indicadores)")
    print(f"Forma de X_test_imputed: {X_test_imputed.shape}")
else:
    print("Error: X_train o X_test no estÃ¡n definidos. Ejecuta los pasos anteriores.")

## 4. Entrenar LightGBM Detector

In [ ]:
if 'X_train_imputed' in locals() and 'y_train' in locals():
    # Inicializar el clasificador LightGBM
    # class_weight='balanced' puede ayudar si hay desbalance entre clases 'normal' y 'anomalia'
    # n_estimators: nÃºmero de Ã¡rboles en el bosque.
    # random_state: para reproducibilidad.
    rf_detector = LGBMClassifier(**MODELO_1_LGBM_PARAMS)

    print("\nEntrenando el modelo LightGBM Detector...")
    rf_detector.fit(X_train_imputed, y_train)
    print("Entrenamiento completado.")

else:
    print("Error: X_train_imputed o y_train no estÃ¡n definidos.")

## 5. Evaluar en test: mÃ©tricas globales

In [ ]:
if 'rf_detector' in locals() and 'X_test_imputed' in locals() and 'y_test' in locals():

    idx_anomalia = list(rf_detector.classes_).index('anomalia')
    y_pred_proba_anomalia = rf_detector.predict_proba(X_test_imputed)[:, idx_anomalia]
    labels_orden  = ['normal', 'anomalia']
    display_names = ['Normal', 'AnomalÃ­a']

    # â”€â”€ Tabla de calibraciÃ³n de umbral â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print("=== CalibraciÃ³n de umbral ===")
    print(f"{'Umbral':>8} {'Accuracy':>10} {'PrecisiÃ³n':>10} {'Recall':>8} {'F1':>8} {'FP':>8} {'FN':>6}")
    print("-" * 65)
    for u in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
        y_u = np.where(y_pred_proba_anomalia >= u, 'anomalia', 'normal')
        cm_u = confusion_matrix(y_test, y_u, labels=labels_orden)
        tn_u, fp_u, fn_u, tp_u = cm_u.ravel()
        acc_u  = accuracy_score(y_test, y_u)
        prec_u = tp_u / (tp_u + fp_u) if (tp_u + fp_u) > 0 else 0
        rec_u  = tp_u / (tp_u + fn_u) if (tp_u + fn_u) > 0 else 0
        f1_u   = 2 * prec_u * rec_u / (prec_u + rec_u) if (prec_u + rec_u) > 0 else 0
        marca  = " â† activo" if abs(u - UMBRAL_DETECCION) < 0.001 else ""
        print(f"{u:>8.2f} {acc_u*100:>9.2f}% {prec_u*100:>9.2f}% {rec_u*100:>7.2f}% {f1_u:>8.4f} {fp_u:>8d} {fn_u:>6d}{marca}")
    print()

    # â”€â”€ PredicciÃ³n con el umbral configurado â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    y_pred_detector = np.where(y_pred_proba_anomalia >= UMBRAL_DETECCION, 'anomalia', 'normal')

    print(f"--- EvaluaciÃ³n del Modelo Detector (umbral={UMBRAL_DETECCION}) ---")
    accuracy = accuracy_score(y_test, y_pred_detector)
    print(f"Accuracy: {accuracy:.4f}")

    print("\nMatriz de ConfusiÃ³n:")
    cm = confusion_matrix(y_test, y_pred_detector, labels=labels_orden)
    print(cm)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=display_names, yticklabels=display_names)
    plt.xlabel('PredicciÃ³n')
    plt.ylabel('Valor Real')
    plt.title(f'Matriz de ConfusiÃ³n - Detector (umbral={UMBRAL_DETECCION})')
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = cm.ravel()
    print(f"\nAnÃ¡lisis Adicional del Detector:")
    print(f"  - Verdaderos Positivos (VP):  {tp}")
    print(f"  - Falsos Negativos (FN):      {fn}")
    print(f"  - Falsos Positivos (FP):      {fp}")
    print(f"  - Verdaderos Negativos (TN):  {tn}")

    print("\nReporte de ClasificaciÃ³n:")
    print(classification_report(y_test, y_pred_detector,
                                labels=labels_orden, target_names=display_names))

    y_bin = (y_test == 'anomalia').astype(int)
    roc_auc = roc_auc_score(y_bin, y_pred_proba_anomalia)
    pr_auc  = average_precision_score(y_bin, y_pred_proba_anomalia)
    print(f"ROC AUC Score:                         {roc_auc:.4f}")
    print(f"Precision-Recall AUC (Average Precision): {pr_auc:.4f}")

else:
    print("Error: El modelo 'rf_detector' no estÃ¡ entrenado o los datos de prueba no estÃ¡n disponibles.")


In [ ]:
# â”€â”€ DiagnÃ³stico FP: Â¿de dÃ³nde vienen los falsos positivos? â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Busca quÃ© sensores, horas y features dominan los FP para localizar la causa.

if ('rf_detector' in locals() and 'X_test_imputed' in locals() and
        'y_test' in locals() and 'y_pred_detector' in locals()):

    y_pred_s  = pd.Series(y_pred_detector, index=X_test.index)
    y_real_s  = pd.Series(y_test.values,   index=X_test.index)

    mask_fp   = (y_pred_s == 'anomalia') & (y_real_s == 'normal')
    mask_tn   = (y_pred_s == 'normal')   & (y_real_s == 'normal')

    print(f"FP totales: {mask_fp.sum():,}  |  TN totales: {mask_tn.sum():,}")

    # â”€â”€ 1. FP por hora del dÃ­a â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    hora_test = df_trabajo.loc[X_test.index, 'Hora']
    fp_by_hora = hora_test[mask_fp].value_counts().sort_index()
    tn_by_hora = hora_test[mask_tn].value_counts().sort_index()
    rate_by_hora = (fp_by_hora / (fp_by_hora + tn_by_hora) * 100).round(1)

    print("\n=== FP rate por hora (% de normales mal clasificados) ===")
    print(rate_by_hora.to_string())

    # â”€â”€ 2. FP por mes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    mes_test = df_trabajo.loc[X_test.index, 'Mes']
    fp_by_mes = hora_test[mask_fp].index  # reusar index
    fp_mes = mes_test[mask_fp].value_counts().sort_index()
    tn_mes = mes_test[mask_tn].value_counts().sort_index()
    rate_mes = (fp_mes / (fp_mes + tn_mes) * 100).round(1)
    print("\n=== FP rate por mes ===")
    print(rate_mes.to_string())

    # â”€â”€ 3. Features ctx mÃ¡s extremas en FP vs TN â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if isinstance(X_test_imputed, np.ndarray):
        try:
            feat_names = list(imputer.get_feature_names_out(X_test.columns))
        except Exception:
            feat_names = [f'f{i}' for i in range(X_test_imputed.shape[1])]
        df_imp = pd.DataFrame(X_test_imputed, index=X_test.index, columns=feat_names)
    else:
        df_imp = X_test_imputed

    ctx_cols = [c for c in df_imp.columns if '_dev_ctx' in c or '_zscore_ctx' in c]
    if ctx_cols:
        mean_fp = df_imp.loc[mask_fp, ctx_cols].mean()
        mean_tn = df_imp.loc[mask_tn, ctx_cols].mean()
        diff    = (mean_fp - mean_tn).abs().sort_values(ascending=False)
        print("\n=== Top 10 ctx features con mayor diferencia FP vs TN ===")
        print(diff.head(10).round(3).to_string())

    # â”€â”€ 4. Features raw del sensor mÃ¡s activadas en FP â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    sensor_consec_cols = [c for c in X_test.columns if '_consecutive_unchanged' in c]
    if sensor_consec_cols:
        fp_consec = X_test.loc[mask_fp, sensor_consec_cols].mean().sort_values(ascending=False)
        print("\n=== consecutive_unchanged medio en FP (Â¿sensor quiet dominant?) ===")
        print(fp_consec.head(9).round(2).to_string())

else:
    print("Ejecuta primero las celdas de entrenamiento y evaluaciÃ³n.")


## 5b. Post-procesado: regla determinista para sensor atascado
Si un sensor lleva `consecutive_unchanged >= 5` minutos con el mismo valor **y** su `rstd_30m` supera un umbral mÃ­nimo (sensor normalmente activo), forzamos la predicciÃ³n a `anomalia` independientemente del modelo.

Esto sigue el enfoque hÃ­brido estÃ¡ndar en literatura: reglas para anomalÃ­as deterministas, ML para anomalÃ­as complejas.

In [ ]:
# Post-procesado: regla determinista para sensor atascado
# Solo se aplica a XCO2I, PCO2EXT, PVV â€” Ãºnicos sensores con buena ratio TP/FP
# PVV excluye valor ~0 (calma natural)

if ('y_pred_detector' in locals() and 'X_test' in locals() and 'df_trabajo' in locals()):

    mask_regla = pd.Series(False, index=X_test.index)

    for col in STUCK_SENSORES_REGLA:
        col_unchanged = f'{col}_consecutive_unchanged'
        if col_unchanged not in df_trabajo.columns:
            continue

        consecutive = df_trabajo.loc[X_test.index, col_unchanged]
        mask_sensor = consecutive >= STUCK_CONSECUTIVOS_MIN

        if col in STUCK_EXCLUIR_VALOR_CERO:
            valor_actual = df_trabajo.loc[X_test.index, col]
            mask_sensor  = mask_sensor & (valor_actual > 0.5)

        mask_regla |= mask_sensor

    # Aplicar la regla sobre y_pred_detector
    y_pred_series = pd.Series(y_pred_detector, index=X_test.index)
    n_antes    = (y_pred_series == 'anomalia').sum()
    y_pred_series[mask_regla] = 'anomalia'
    n_forzados = (y_pred_series == 'anomalia').sum() - n_antes
    y_pred_detector = y_pred_series.values

    print(f"Regla sensor atascado: {n_forzados} predicciones adicionales forzadas a 'anomalia'")
    print(f"  Total anomalÃ­as en test: {n_antes} â†’ {n_antes + n_forzados}")

    # â”€â”€ DiagnÃ³stico de falsos positivos â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    y_real_series = pd.Series(y_test.values, index=X_test.index)
    n_fp = (mask_regla & (y_real_series == 'normal')).sum()
    n_tp = (mask_regla & (y_real_series == 'anomalia')).sum()

    print(f"\n--- DiagnÃ³stico de la regla ---")
    print(f"  Filas donde disparÃ³ la regla:           {mask_regla.sum()}")
    print(f"  â†’ Verdaderos positivos (eran anomalÃ­a): {n_tp}")
    print(f"  â†’ Falsos positivos (eran normales):     {n_fp}")
    if mask_regla.sum() > 0:
        print(f"  PrecisiÃ³n de la regla:                  {n_tp / mask_regla.sum() * 100:.1f}%")

    print(f"\n  Detalle por sensor:")
    for col in STUCK_SENSORES_REGLA:
        col_unchanged = f'{col}_consecutive_unchanged'
        if col_unchanged not in df_trabajo.columns:
            continue
        consecutive = df_trabajo.loc[X_test.index, col_unchanged]
        mask_s = consecutive >= STUCK_CONSECUTIVOS_MIN
        if col in STUCK_EXCLUIR_VALOR_CERO:
            mask_s = mask_s & (df_trabajo.loc[X_test.index, col] > 0.5)
        tp = (mask_s & (y_real_series == 'anomalia')).sum()
        fp = (mask_s & (y_real_series == 'normal')).sum()
        print(f"    {col}: {tp} TP / {fp} FP")
else:
    print("Variables no disponibles para aplicar la regla.")

## 5c. Clasificador binario especÃ­fico para Sensor Atascado
Entrenamos un Random Forest binario secundario usando **solo las features M3** (rstd_5m, n_zero_diff_10m, consecutive_unchanged, stuck_x_prad).

El target es binario: `1` si la fila es `Sensor Atascado`, `0` si es `normal`.

Este clasificador actÃºa en paralelo al Modelo 1: si alguno de los dos predice `anomalia`, la fila se marca como anomalÃ­a. El resultado combinado se evalÃºa en la celda siguiente.

In [ ]:
# â”€â”€ 5c. Clasificador binario para Sensor Atascado â€” DIAGNÃ“STICO (no aplicado) â”€
# Entrenamos el clasificador stuck para medir su potencial, pero NO lo aplicamos
# porque su precisiÃ³n en XCO2I es ~7%: 12 FP por cada TP. El resultado combinado
# degrada la precisiÃ³n global de 99.76% a 64%.
# La mejora de SA se aborda mediante baselines contextuales (M7) y regla XHINV (5d).

M3_SUFIJOS = ['_rstd_5m', '_n_zero_diff_10m', '_consecutive_unchanged',
              '_stuck_x_prad', '_max_accel_5m', '_stuck_vs_neighbors',
              '_pre_vs_current_rstd']
cols_m3 = [c for c in df_trabajo.columns if any(c.endswith(s) for s in M3_SUFIJOS)]
cols_contexto = [c for c in ['Mes', 'Hora', 'DiaSemana'] if c in df_trabajo.columns]
cols_stuck_features = cols_m3 + cols_contexto

print(f"Features stuck classifier: {len(cols_stuck_features)} "
      f"({len(cols_m3)} M3 + {len(cols_contexto)} temporales)")

if len(cols_m3) == 0:
    print("ERROR: No se encontraron features M3. Re-ejecuta 03_features.ipynb.")
else:
    etiquetas_det  = df_trabajo['etiqueta_deteccion']
    etiquetas_tipo = df_trabajo['etiqueta_tipo_anomalia']

    mask_sa_train = (etiquetas_tipo.iloc[train_idx_final] == 'Sensor Atascado')
    idx_sa_train  = df_trabajo.index[train_idx_final][mask_sa_train]

    mask_normal_train = (etiquetas_det.iloc[train_idx_final] == 'normal')
    idx_normal_all    = df_trabajo.index[train_idx_final][mask_normal_train]

    consec_cols = [c for c in cols_m3 if c.endswith('_consecutive_unchanged')]
    if consec_cols:
        max_consec = df_trabajo.loc[idx_normal_all, consec_cols].max(axis=1)
        idx_hard_neg = idx_normal_all[max_consec >= 3]
    else:
        idx_hard_neg = idx_normal_all

    n_pos = len(idx_sa_train)
    n_neg_max = min(len(idx_hard_neg), n_pos * 5)
    rng = np.random.default_rng(42)
    idx_neg_sample = rng.choice(idx_hard_neg, size=n_neg_max, replace=False)

    idx_train_stuck = np.concatenate([idx_sa_train, idx_neg_sample])
    y_stuck_train   = np.concatenate([
        np.ones(len(idx_sa_train), dtype=int),
        np.zeros(len(idx_neg_sample), dtype=int),
    ])

    print(f"Train: {len(idx_sa_train):,} SA positivos + {len(idx_neg_sample):,} negativos estables")

    idx_test_stuck = df_trabajo.index[test_idx_final]
    X_stuck_train  = df_trabajo.loc[idx_train_stuck, cols_stuck_features].copy()
    X_stuck_test   = df_trabajo.loc[idx_test_stuck,  cols_stuck_features].copy()

    from sklearn.impute import SimpleImputer as _SI
    imp_stuck = _SI(strategy='median')
    X_stuck_train_imp = imp_stuck.fit_transform(X_stuck_train)
    X_stuck_test_imp  = imp_stuck.transform(X_stuck_test)

    from sklearn.ensemble import RandomForestClassifier as _RFC
    rf_stuck = _RFC(n_estimators=200, random_state=42, class_weight='balanced',
                    n_jobs=-1, max_depth=10, min_samples_leaf=3)
    rf_stuck.fit(X_stuck_train_imp, y_stuck_train)

    proba_stuck = rf_stuck.predict_proba(X_stuck_test_imp)[:, 1]
    etiquetas_test_tipo = df_trabajo.loc[idx_test_stuck, 'etiqueta_tipo_anomalia']

    print("\n--- CalibraciÃ³n de umbral (solo diagnÃ³stico â€” no se aplica) ---")
    for u in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        mask_u   = proba_stuck >= u
        tp_u     = (mask_u & (etiquetas_test_tipo == 'Sensor Atascado')).sum()
        fp_u     = (mask_u & (y_test == 'normal')).sum()
        total_sa = (etiquetas_test_tipo == 'Sensor Atascado').sum()
        prec_u   = tp_u / mask_u.sum() * 100 if mask_u.sum() > 0 else 0
        rec_u    = tp_u / total_sa * 100 if total_sa > 0 else 0
        print(f"  umbral={u:.1f}: {mask_u.sum():6d} disparos | "
              f"TP={tp_u:3d} ({rec_u:.1f}% recall SA) | "
              f"FP={fp_u:6d} | PrecisiÃ³n={prec_u:.1f}%")

    print("\nâš  Stuck classifier NO aplicado â€” precisiÃ³n ~7% (12 FP por TP).")
    print("  Mejora de SA abordada por baselines ctx (M7) + regla XHINV independiente (5d).")
    # y_pred_detector no se modifica aquÃ­


In [ ]:
# â”€â”€ 5d. Regla contextual XHINV â€” aplicada independientemente â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# XHINV: 878 disparos, TP=860, FP=18 â†’ precisiÃ³n 97.9% (diagnÃ³stico previo).
# Se aplica directamente sin guardiÃ¡n global porque su precisiÃ³n es suficiente.
#
# PRGINT desactivada: 2 disparos, 0 TP, 2 FP en ejecucion 002 -> no aporta.
# PTEXT se mantiene inactiva: demasiados FP para usarla como regla contextual.

REGLAS_CONTEXTO_CRUZADO = {
    'XHINV': {
        'ref_std_col': 'XTINV_rstd_5m',
        'ref_std_min': 0.08,
        'consec_min':  40,     # humedad puede saturarse horas â†’ SA necesita 40+ min
        'valor_max':   98.0,   # excluir saturaciÃ³n natural (â‰¥98% HR es normal)
        'aplicar':     True,   # precisiÃ³n 97.9% â€” siempre activa
    },
    'PTEXT': {
        'ref_std_col': 'XTINV_rstd_5m',
        'ref_std_min': 0.08,
        'consec_min':  10,
        'aplicar':     False,  # precisiÃ³n 9.7% â€” desactivada
    },
    'PRGINT': {
        'ref_std_col':        'PRAD_rstd_5m',
        'ref_std_min':        1.0,
        'consec_min':         5,
        'excluir_valor_cero': True,
        'valor_min':          5.0,
        'aplicar':            False,   # desactivada: 0 TP, 2 FP en ejecucion 002
    },
}

if ('y_pred_detector' in locals() and 'X_test' in locals() and 'df_trabajo' in locals()):
    y_real_series = pd.Series(y_test.values, index=X_test.index)
    y_pred_series = pd.Series(y_pred_detector, index=X_test.index)

    print("=== 5d. Regla contextual cruzada â€” diagnÃ³stico y aplicaciÃ³n selectiva ===")
    total_forzados = 0

    for sensor, cfg in REGLAS_CONTEXTO_CRUZADO.items():
        col_consec  = f"{sensor}_consecutive_unchanged"
        col_ref_std = cfg['ref_std_col']

        if col_consec not in df_trabajo.columns or col_ref_std not in df_trabajo.columns:
            print(f"  {sensor}: columnas no disponibles, omitido")
            continue

        consec  = df_trabajo.loc[X_test.index, col_consec]
        ref_std = df_trabajo.loc[X_test.index, col_ref_std]
        mask_s  = (consec >= cfg['consec_min']) & (ref_std >= cfg['ref_std_min'])

        if cfg.get('excluir_valor_cero') and sensor in df_trabajo.columns:
            mask_s = mask_s & (df_trabajo.loc[X_test.index, sensor] >= cfg.get('valor_min', 1.0))
        if 'valor_max' in cfg and sensor in df_trabajo.columns:
            mask_s = mask_s & (df_trabajo.loc[X_test.index, sensor] < cfg['valor_max'])

        tp   = (mask_s & (y_real_series == 'anomalia')).sum()
        fp   = (mask_s & (y_real_series == 'normal')).sum()
        prec = tp / mask_s.sum() * 100 if mask_s.sum() > 0 else 0
        estado = "âœ“ ACTIVA" if cfg['aplicar'] else "âœ— inactiva"
        print(f"  {sensor} [{estado}]: {mask_s.sum():,} disparos | "
              f"TP={tp} / FP={fp} | PrecisiÃ³n={prec:.1f}%")

        if cfg['aplicar']:
            n_antes = (y_pred_series == 'anomalia').sum()
            y_pred_series[mask_s] = 'anomalia'
            n_nuevos = (y_pred_series == 'anomalia').sum() - n_antes
            total_forzados += n_nuevos
            print(f"    â†’ {n_nuevos} predicciones adicionales forzadas a 'anomalia'")

    y_pred_detector = y_pred_series.values
    print(f"\nTotal predicciones forzadas por reglas ctx activas: {total_forzados}")

    # Recall SA por sensor post-regla
    tipos_test = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    pred_post  = pd.Series(y_pred_detector, index=X_test.index)
    cols_consec_all = [f'{c}_consecutive_unchanged' for c in COLUMNAS_SENSORES
                       if f'{c}_consecutive_unchanged' in df_trabajo.columns]

    print("\n=== Recall SA por sensor DESPUES de reglas ctx ===")
    mask_sa_s  = tipos_test == 'Sensor Atascado'
    idx_sa_all = X_test.index[mask_sa_s]
    if 'sensor_anomalia' in df_trabajo.columns:
        sensor_real_all = df_trabajo.loc[idx_sa_all, 'sensor_anomalia'].fillna('sin_sensor')
        print("  Fuente sensor: sensor_anomalia")
    elif cols_consec_all:
        df_sa_tmp = df_trabajo.loc[idx_sa_all, cols_consec_all]
        sensor_real_all = df_sa_tmp.idxmax(axis=1).str.replace('_consecutive_unchanged', '')
        print("  Fuente sensor: idxmax(consecutive_unchanged) [fallback]")
    else:
        sensor_real_all = pd.Series('sin_sensor', index=idx_sa_all)
        print("  Fuente sensor: no disponible")

    for sensor in ['XHINV', 'PTEXT', 'PRGINT']:
        idx_sa_sensor = idx_sa_all[sensor_real_all.eq(sensor).to_numpy()]
        if len(idx_sa_sensor) == 0:
            print(f"  {sensor}: 0 filas SA reales en test")
            continue
        detectados = (pred_post.loc[idx_sa_sensor] == 'anomalia').sum()
        recall     = detectados / len(idx_sa_sensor) * 100
        print(f"  {sensor}: {detectados}/{len(idx_sa_sensor)} -> recall={recall:.1f}%")

else:
    print("Variables no disponibles.")


## 5e. Regla determinista: Valores Fuera de Rango
Si el valor bruto del sensor supera el rango fisico de hardware (`RANGOS_FISICOS`),
forzamos la prediccion a `anomalia` directamente.

In [ ]:
# -- 5e. Regla determinista: Valores Fuera de Rango
if ('y_pred_detector' in locals() and 'X_test' in locals() and 'df_trabajo' in locals()):
    y_real_series = pd.Series(y_test.values, index=X_test.index)
    y_pred_series = pd.Series(y_pred_detector, index=X_test.index)
    mask_oor = pd.Series(False, index=X_test.index)

    print('=== 5e. Regla Fuera de Rango ===')
    for sensor, rango in RANGOS_FISICOS.items():
        if sensor not in df_trabajo.columns:
            continue
        valores = df_trabajo.loc[X_test.index, sensor]
        mask_s  = ((valores < rango['min']) | (valores > rango['max'])) & valores.notna()
        if mask_s.sum() == 0:
            continue
        tp   = (mask_s & (y_real_series == 'anomalia')).sum()
        fp   = (mask_s & (y_real_series == 'normal')).sum()
        prec = tp / mask_s.sum() * 100 if mask_s.sum() > 0 else 0
        print(f'  {sensor}: {mask_s.sum():,} disparos | TP={tp} / FP={fp} | Precision={prec:.1f}%')
        mask_oor |= mask_s

    n_antes  = (y_pred_series == 'anomalia').sum()
    y_pred_series[mask_oor] = 'anomalia'
    n_nuevos = (y_pred_series == 'anomalia').sum() - n_antes
    tp_total   = (mask_oor & (y_real_series == 'anomalia')).sum()
    fp_total   = (mask_oor & (y_real_series == 'normal')).sum()
    prec_total = tp_total / mask_oor.sum() * 100 if mask_oor.sum() > 0 else 0
    print(f'\nTotal OOR: {mask_oor.sum():,} | TP={tp_total} / FP={fp_total} | Precision={prec_total:.1f}%')
    print(f'Predicciones nuevas forzadas: {n_nuevos}')
    y_pred_detector = y_pred_series.values
else:
    print('Variables no disponibles.')


## 5f. Diagnostico: Datos Faltantes
Se evalua el potencial de una regla NaN, pero NO se aplica.
La regla genera demasiados FP por gaps naturales/comunicacion.

In [ ]:
# -- 5f. DIAGNOSTICO (no aplicado): Datos Faltantes
# La regla NaN genera muchos FP por gaps naturales/comunicaciÃ³n.
# Se deja solo como diagnÃ³stico y NO modifica y_pred_detector.
if ("y_pred_detector" in locals() and "X_test" in locals() and "df_trabajo" in locals()):
    y_real_series = pd.Series(y_test.values, index=X_test.index)
    mask_nan = pd.Series(False, index=X_test.index)
    print("=== 5f. DIAGNOSTICO Datos Faltantes (regla NO aplicada) ===")
    for sensor in COLUMNAS_SENSORES:
        if sensor not in df_trabajo.columns:
            continue
        mask_s = df_trabajo.loc[X_test.index, sensor].isna()
        if mask_s.sum() == 0:
            continue
        tp   = (mask_s & (y_real_series == "anomalia")).sum()
        fp   = (mask_s & (y_real_series == "normal")).sum()
        prec = tp / mask_s.sum() * 100 if mask_s.sum() > 0 else 0
        print(f"  {sensor}: {mask_s.sum():,} NaN | TP={tp} / FP={fp} | Precision={prec:.1f}%")
        mask_nan |= mask_s

    tp_t = (mask_nan & (y_real_series == "anomalia")).sum()
    fp_t = (mask_nan & (y_real_series == "normal")).sum()
    prec_t = tp_t / mask_nan.sum() * 100 if mask_nan.sum() > 0 else 0
    print()
    print(f"Total NaN: {mask_nan.sum():,} | TP={tp_t} / FP={fp_t} | Precision={prec_t:.1f}%")
    print("Regla NO aplicada para mantener coherencia con la evaluaciÃ³n final.")
else:
    print("Variables no disponibles.")


## 5g. Metricas finales combinadas
Evaluacion global tras aplicar: RF(umbral=config) + regla SA + reglas ctx activas + regla OOR. Regla NaN solo diagnostico.

In [ ]:
# -- 5g. Metricas finales combinadas
if 'y_pred_detector' in locals() and 'y_test' in locals():
    labels_orden  = ['normal', 'anomalia']
    display_names = ['Normal', 'Anomalia']

    cm = confusion_matrix(y_test, y_pred_detector, labels=labels_orden)
    tn, fp, fn, tp = cm.ravel()
    prec_a = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec_a  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_a   = 2 * prec_a * rec_a / (prec_a + rec_a) if (prec_a + rec_a) > 0 else 0

    print('=' * 60)
    print('METRICAS FINALES (LightGBM@umbral + reglas SA + ctx activas + OOR)')
    print('=' * 60)
    print(f'Accuracy : {accuracy_score(y_test, y_pred_detector)*100:.2f}%')
    print(f'TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
    print(f'Anomalia -- Precision={prec_a*100:.2f}%  Recall={rec_a*100:.2f}%  F1={f1_a:.4f}')
    print()
    print(classification_report(y_test, y_pred_detector,
                                labels=labels_orden, target_names=display_names))

    tipos_test = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    pred_final = pd.Series(y_pred_detector, index=X_test.index)

    print('=== Deteccion por tipo de anomalia (final) ===')
    for tipo in ['Datos Faltantes', 'Valores Fuera de Rango', 'Ruido',
                 'Sensor Atascado', 'Desviacion de Correlacion', 'Contextual']:
        mask_tipo = tipos_test == tipo
        if mask_tipo.sum() == 0:
            continue
        detectados = (pred_final[mask_tipo] == 'anomalia').sum()
        total = mask_tipo.sum()
        print(f'  {tipo}: {detectados}/{total} -> {detectados/total*100:.1f}%')
else:
    print('Variables no disponibles.')


## 6. Importancia de features
Â¿QuÃ© variables son mÃ¡s relevantes para detectar anomalÃ­as?

In [ ]:
if ('rf_detector' in locals() and
    hasattr(rf_detector, 'feature_importances_') and
    'X_train_imputed' in locals()): # Verificar X_train_imputed

    if not isinstance(X_train_imputed, pd.DataFrame):
        print("Error: X_train_imputed no es un DataFrame de Pandas. No se pueden obtener nombres de caracterÃ­sticas.")
        print("Esto puede suceder si la construcciÃ³n de feature_names_imputed fallÃ³ en el paso de imputaciÃ³n.")
    else:
        importances = rf_detector.feature_importances_
        feature_names = X_train_imputed.columns
        
        # Verificar la longitud para evitar el error
        if len(feature_names) == len(importances):
            # Crear un DataFrame para visualizar mejor
            feature_importance_df = pd.DataFrame({'CaracterÃ­stica': feature_names, 'Importancia': importances})
            feature_importance_df = feature_importance_df.sort_values(by='Importancia', ascending=False)
            
            print("\n--- Importancia de las CaracterÃ­sticas (Detector de AnomalÃ­as) ---")
            print(feature_importance_df.to_string())
            
            # Graficar la importancia de las caracterÃ­sticas
            plt.figure(figsize=(12, max(6, len(feature_names) * 0.4))) 
            plt.barh(feature_importance_df['CaracterÃ­stica'], feature_importance_df['Importancia'], color='mediumpurple')
            plt.xlabel("Importancia Relativa")
            plt.ylabel("CaracterÃ­stica")
            plt.title("Importancia de las CaracterÃ­sticas - Detector de AnomalÃ­as (LightGBM)")
            plt.gca().invert_yaxis() 
            plt.tight_layout() 
            plt.show()
        else:
            print(f"Error de coincidencia de longitud: Nombres de caracterÃ­sticas ({len(feature_names)}) vs. Importancias ({len(importances)}).")
            print("AsegÃºrate de que X_train_imputed tenga los nombres de columna correctos despuÃ©s de la imputaciÃ³n.")

else:
    print("Error: No se pudo calcular la importancia de las caracterÃ­sticas.")
    print("AsegÃºrate de que 'rf_detector' estÃ© entrenado y 'X_train_imputed' (como DataFrame con columnas) estÃ© disponible.")

## 7. Desglose por tipo de anomalÃ­a
Â¿CuÃ¡ntas anomalÃ­as de cada tipo detecta correctamente el modelo?

In [ ]:
if ('df_trabajo' in locals() and
    'X_test' in locals() and
    'y_test' in locals() and
    'y_pred_detector' in locals()):

    # Obtener los tipos de anomalÃ­a reales para el conjunto de prueba
    # X_test.index contiene los Ã­ndices originales de df_trabajo para las filas de prueba
    true_anomaly_types_in_test_set = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    
    # Crear el DataFrame para anÃ¡lisis
    df_eval_detector = pd.DataFrame({
        'indice_original': X_test.index,
        'y_real_binaria': y_test,          # 'normal' o 'anomalia'
        'y_pred_detector': y_pred_detector, # 'normal' o 'anomalia' predicho
        'tipo_anomalia_real': true_anomaly_types_in_test_set
    })

    print("\n--- Desglose de DetecciÃ³n del Modelo 1 por Tipo Real de AnomalÃ­a (en Conjunto de Prueba) ---")

    # Filtrar solo las filas que SON REALMENTE ANOMALÃAS
    df_anomalias_reales_en_test = df_eval_detector[df_eval_detector['y_real_binaria'] == 'anomalia']

    if df_anomalias_reales_en_test.empty:
        print("No hay anomalÃ­as reales en el conjunto de prueba para analizar.")
    else:
        # Agrupar por el tipo real de anomalÃ­a y contar cuÃ¡ntas fueron detectadas
        detection_summary = df_anomalias_reales_en_test.groupby('tipo_anomalia_real').agg(
            total_instancias_reales_en_test = pd.NamedAgg(column='y_pred_detector', aggfunc='count'),
            correctamente_detectadas_como_anomalia = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'anomalia').sum()),
            no_detectadas_por_el_detector_FN = pd.NamedAgg(column='y_pred_detector', aggfunc=lambda x: (x == 'normal').sum())
        ).reset_index()

        # Calcular tasa de detecciÃ³n por tipo
        detection_summary['tasa_deteccion_%'] = (
            detection_summary['correctamente_detectadas_como_anomalia'] / 
            detection_summary['total_instancias_reales_en_test'] * 100
        )
        
        # Ordenar por tasa de detecciÃ³n o total de instancias para mejor visualizaciÃ³n
        detection_summary = detection_summary.sort_values(by='tasa_deteccion_%', ascending=False)

        print("\nResumen de detecciÃ³n por tipo de anomalÃ­a real:")
        print(detection_summary.to_string())
        
        # Graficar la tasa de detecciÃ³n por tipo de anomalÃ­a
        if not detection_summary.empty:
            plt.figure(figsize=(12, max(6, len(detection_summary) * 0.5)))
            
            detection_summary.set_index('tipo_anomalia_real')[['correctamente_detectadas_como_anomalia', 'no_detectadas_por_el_detector_FN']].plot(
                kind='barh', 
                stacked=True, 
                figsize=(12, max(6, len(detection_summary) * 0.6)),
                colormap='viridis'
            )
            plt.title('Rendimiento del Detector por Tipo de AnomalÃ­a Real (en Conjunto de Prueba)')
            plt.xlabel('NÃºmero de Instancias')
            plt.ylabel('Tipo de AnomalÃ­a Real')
            plt.legend(title='Resultado DetecciÃ³n', loc='lower right')
            plt.tight_layout()
            plt.show()

            plt.figure(figsize=(10, max(5, len(detection_summary) * 0.5) ))
            sns.barplot(x='tasa_deteccion_%', y='tipo_anomalia_real', data=detection_summary, palette='magma', hue='tipo_anomalia_real', dodge=False, legend=False)
            plt.title('Tasa de DetecciÃ³n del Modelo 1 por Tipo de AnomalÃ­a Real (%)')
            plt.xlabel('Tasa de DetecciÃ³n (%)')
            plt.ylabel('Tipo de AnomalÃ­a Real')
            plt.xlim(0, 105)
            for index, row in detection_summary.iterrows():
                plt.text(row['tasa_deteccion_%'] + 1,
                        index,
                        f"{row['tasa_deteccion_%']:.1f}%", 
                        color='black', 
                        ha="left", 
                        va="center")
            plt.tight_layout()
            plt.show()

else:
    print("Error: No se pueden generar los datos de desglose.")
    print("AsegÃºrate de que 'df_trabajo', 'X_test', 'y_test', y 'y_pred_detector' estÃ©n disponibles.")

In [ ]:
# Diagnostico DesCor: FN por par de sensores
_reqs_dc = ['df_eval_detector', 'df_trabajo', 'INYECCION']
_missing_dc = [v for v in _reqs_dc if v not in globals()]
if _missing_dc:
    print(f'Faltan variables: {_missing_dc}. Ejecuta las celdas anteriores.')
else:
    mask_dc_ev = df_eval_detector['tipo_anomalia_real'] == 'Desviacion de Correlacion'
    df_dc = df_eval_detector[mask_dc_ev].copy()

    if df_dc.empty:
        print('No hay instancias DesCor en el conjunto de evaluacion.')
    else:
        df_dc['detectado'] = df_dc['y_pred_detector'] == 'anomalia'

        pares_test = INYECCION.get('desviacion_correlacion', {}).get('instancias_por_par_temporal', {}).get('test', {})
        sensor_to_par = {sA: f'({sA}, {sB})' for (sA, sB) in pares_test.keys()}

        if 'sensor_anomalia' in df_trabajo.columns:
            df_dc['sensor_mod'] = df_trabajo.loc[df_dc['indice_original'].values, 'sensor_anomalia'].values
            df_dc['par'] = df_dc['sensor_mod'].map(sensor_to_par).fillna('desconocido')
        else:
            df_dc['par'] = 'sin_trazabilidad'

        print('=== DesCor: recall por par de sensores ===')
        resumen = (df_dc.groupby('par')
                       .agg(total=('detectado', 'count'),
                            detectados=('detectado', 'sum'))
                       .assign(FN=lambda d: d['total'] - d['detectados'],
                               recall_pct=lambda d: (d['detectados'] / d['total'] * 100).round(1)))
        print(resumen.to_string())

        total_dc = len(df_dc)
        total_det = int(df_dc['detectado'].sum())
        print(f'Total: {total_dc} | Detectados: {total_det} | FN: {total_dc - total_det} | Recall: {total_det/total_dc*100:.1f}%')

        if 'Hora' in df_trabajo.columns:
            df_dc['Hora'] = df_trabajo.loc[df_dc['indice_original'].values, 'Hora'].values
            fn_hora = df_dc[~df_dc['detectado']].groupby(['par', 'Hora']).size().rename('FN')
            if not fn_hora.empty:
                print('=== FN DesCor por par y hora ===')
                print(fn_hora.to_string())

In [ ]:
# Diagnostico trazabilidad SA por split
_reqs_trace = ['df_trabajo', 'X_train', 'X_test']
_missing_trace = [v for v in _reqs_trace if v not in globals()]
if _missing_trace:
    print(f"Faltan variables para trazabilidad: {_missing_trace}")
elif 'sensor_anomalia' not in df_trabajo.columns:
    print("sensor_anomalia no disponible. Ejecuta 02 -> 03 con la nueva trazabilidad.")
else:
    print("=== Sensor Atascado real por sensor_anomalia ===")
    for nombre, idx_split in [('train', X_train.index), ('test', X_test.index)]:
        mask_sa_split = df_trabajo.loc[idx_split, 'etiqueta_tipo_anomalia'].eq('Sensor Atascado')
        dist = (
            df_trabajo.loc[idx_split[mask_sa_split], 'sensor_anomalia']
            .fillna('sin_sensor')
            .value_counts()
            .sort_index()
        )
        print(f"\n{nombre}: {int(mask_sa_split.sum())} filas SA")
        print(dist.to_string() if len(dist) else 'Sin SA')


In [ ]:
# Diagnostico SA: FN por sensor + visualizacion
_reqs = ['df_trabajo', 'X_test', 'y_test', 'y_pred_detector']
_missing = [v for v in _reqs if v not in globals()]
if _missing:
    print(f"Faltan variables: {_missing}. Ejecuta las celdas anteriores.")
else:
    tipos_test = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    mask_sa    = tipos_test == 'Sensor Atascado'
    idx_sa     = X_test.index[mask_sa]
    print(f"Filas SA en test: {mask_sa.sum()}")

    cols_consec = [f'{col}_consecutive_unchanged' for col in COLUMNAS_SENSORES
                   if f'{col}_consecutive_unchanged' in df_trabajo.columns]

    base_cols = ['Fecha'] + cols_consec
    if 'sensor_anomalia' in df_trabajo.columns:
        base_cols.append('sensor_anomalia')
    if 'anomalia_id' in df_trabajo.columns:
        base_cols.append('anomalia_id')

    df_sa = df_trabajo.loc[idx_sa, base_cols].copy()
    if 'sensor_anomalia' in df_sa.columns:
        df_sa['sensor_stuck'] = df_sa['sensor_anomalia'].fillna('sin_sensor')
        fuente_sensor = 'sensor_anomalia'
    elif cols_consec:
        df_sa['sensor_stuck'] = (
            df_sa[cols_consec].idxmax(axis=1).str.replace('_consecutive_unchanged', '', regex=False)
        )
        fuente_sensor = 'idxmax(consecutive_unchanged) [fallback]'
    else:
        df_sa['sensor_stuck'] = 'sin_sensor'
        fuente_sensor = 'no disponible'

    print(f"Fuente sensor SA: {fuente_sensor}")
    pred_series = pd.Series(y_pred_detector, index=X_test.index)
    df_sa['detectado'] = pred_series.loc[idx_sa].values == 'anomalia'
    df_sa['sensor_stuck_detalle'] = df_sa['sensor_stuck']
    if 'anomalia_id' in df_sa.columns:
        anomalia_str = df_sa['anomalia_id'].fillna('').astype(str)
        mask_ptext = df_sa['sensor_stuck'].eq('PTEXT')
        mask_ptext_direct = mask_ptext & anomalia_str.str.startswith('DIRECT_PTEXT')
        df_sa.loc[mask_ptext_direct, 'sensor_stuck_detalle'] = 'PTEXT_direct_short'
        df_sa.loc[mask_ptext & ~mask_ptext_direct, 'sensor_stuck_detalle'] = 'PTEXT_stuck_standard'

    resumen = df_sa.groupby('sensor_stuck').agg(
        total=('detectado', 'count'),
        detectados=('detectado', 'sum'),
    )
    resumen['FN'] = resumen['total'] - resumen['detectados']
    resumen['recall_%'] = (resumen['detectados'] / resumen['total'] * 100).round(1)
    resumen = resumen.sort_values('recall_%')
    print("\n=== SA: recall por sensor ===")
    print(resumen.to_string())
    print(f"\nTotal SA en test: {len(idx_sa)}  |  "
          f"Detectados: {df_sa['detectado'].sum()}  |  "
          f"FN: {(~df_sa['detectado']).sum()}")

    mask_ptext_sa = df_sa['sensor_stuck'].eq('PTEXT')
    if mask_ptext_sa.any():
        df_ptext_sa = df_sa.loc[mask_ptext_sa].copy()
        resumen_ptext = df_ptext_sa.groupby('sensor_stuck_detalle').agg(
            total=('detectado', 'count'),
            detectados=('detectado', 'sum'),
        )
        resumen_ptext['FN'] = resumen_ptext['total'] - resumen_ptext['detectados']
        resumen_ptext['recall_%'] = (resumen_ptext['detectados'] / resumen_ptext['total'] * 100).round(1)
        resumen_ptext = resumen_ptext.sort_values('recall_%')
        print("\n=== PTEXT SA: detalle por tipo de caso (filas) ===")
        print(resumen_ptext.to_string())

        if 'anomalia_id' in df_ptext_sa.columns:
            seq_ptext = (
                df_ptext_sa.groupby(['sensor_stuck_detalle', 'anomalia_id'])
                .agg(detectado_any=('detectado', 'any'), filas=('detectado', 'count'))
                .reset_index()
            )
            resumen_seq_ptext = seq_ptext.groupby('sensor_stuck_detalle').agg(
                secuencias=('anomalia_id', 'count'),
                sec_detectadas=('detectado_any', 'sum'),
                filas=('filas', 'sum'),
            )
            resumen_seq_ptext['sec_perdidas'] = resumen_seq_ptext['secuencias'] - resumen_seq_ptext['sec_detectadas']
            resumen_seq_ptext['recall_seq_%'] = (resumen_seq_ptext['sec_detectadas'] / resumen_seq_ptext['secuencias'] * 100).round(1)
            resumen_seq_ptext = resumen_seq_ptext.sort_values('recall_seq_%')
            print("\n=== PTEXT SA: detalle por tipo de caso (secuencias) ===")
            print(resumen_seq_ptext.to_string())

    sensores_sa = [s for s in resumen.index.tolist() if s in df_trabajo.columns]
    if sensores_sa:
        n = len(sensores_sa)
        fig, axes = plt.subplots(n, 1, figsize=(18, 3.5 * n), sharex=False)
        if n == 1:
            axes = [axes]

        fechas_test = df_trabajo.loc[X_test.index, 'Fecha'].values
        for ax, sensor in zip(axes, sensores_sa):
            ax.plot(fechas_test, df_trabajo.loc[X_test.index, sensor].values,
                    color='steelblue', alpha=0.35, lw=0.5)

            df_s = df_sa[df_sa['sensor_stuck'] == sensor]
            idx_tp = df_s[df_s['detectado']].index
            idx_fn = df_s[~df_s['detectado']].index

            if len(idx_tp):
                ax.scatter(df_trabajo.loc[idx_tp, 'Fecha'], df_trabajo.loc[idx_tp, sensor],
                           color='green', s=30, label=f'Detectado TP ({len(idx_tp)})', zorder=5)
            if len(idx_fn):
                ax.scatter(df_trabajo.loc[idx_fn, 'Fecha'], df_trabajo.loc[idx_fn, sensor],
                           color='red', s=30, label=f'No detectado FN ({len(idx_fn)})', zorder=5)

            r = resumen.loc[sensor, 'recall_%']
            ax.set_title(f'{sensor} - Recall SA: {r}% (TP={len(idx_tp)}, FN={len(idx_fn)})', fontsize=11)
            ax.legend(fontsize=9, loc='upper right')
            ax.grid(True, alpha=0.3)
            ax.set_ylabel(sensor)

        axes[-1].set_xlabel('Fecha')
        plt.suptitle('Sensor Atascado real: detectados (verde) vs no detectados (rojo) por sensor',
                     fontsize=13, y=1.01)
        plt.tight_layout()
        plt.show()
    else:
        print("No hay sensores SA graficables con columna fisica en df_trabajo.")


## 8. Registrar historial de ejecuciones

In [ ]:
# Append automatico de metricas al historial de ejecuciones
from datetime import datetime
from pathlib import Path

_reqs_hist = ['PROJECT_DIR', 'df_trabajo', 'X_test', 'y_test', 'y_pred_detector']
_missing_hist = [v for v in _reqs_hist if v not in globals()]

if _missing_hist:
    print(f"No se puede registrar historial. Faltan variables: {_missing_hist}")
else:
    hist_path = Path(PROJECT_DIR) / 'historial_ejecuciones_04b_modelo_deteccion_lightgbm_full_v3.txt'
    if not hist_path.exists():
        hist_path.write_text(
            'Historial de ejecuciones - 04b_modelo_deteccion_lightgbm.ipynb\n'
            f'Proyecto: {Path(PROJECT_DIR).name}\n\n',
            encoding='utf-8'
        )

    hist_txt = hist_path.read_text(encoding='utf-8', errors='ignore')
    num_ejec = hist_txt.count('EJECUCION ') + 1
    fecha_registro = datetime.now().astimezone().strftime('%Y-%m-%d %H:%M:%S %z')

    labels_orden = ['normal', 'anomalia']
    cm_hist = confusion_matrix(y_test, y_pred_detector, labels=labels_orden)
    tn, fp, fn, tp = cm_hist.ravel()
    acc = accuracy_score(y_test, y_pred_detector) * 100
    prec = tp / (tp + fp) * 100 if (tp + fp) else 0.0
    rec = tp / (tp + fn) * 100 if (tp + fn) else 0.0
    f1_hist = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0

    tipos_test = df_trabajo.loc[X_test.index, 'etiqueta_tipo_anomalia']
    pred_final = pd.Series(y_pred_detector, index=X_test.index)
    lineas_tipo = []
    for tipo in TIPOS_ANOMALIA:
        mask_tipo = tipos_test == tipo
        total_tipo = int(mask_tipo.sum())
        if total_tipo == 0:
            continue
        det_tipo = int((pred_final.loc[mask_tipo] == 'anomalia').sum())
        tasa_tipo = det_tipo / total_tipo * 100
        lineas_tipo.append(f"{tipo}: {det_tipo}/{total_tipo} -> {tasa_tipo:.1f}%")

    lineas_sa = []
    lineas_ptext_sa = []
    mask_sa = tipos_test == 'Sensor Atascado'
    idx_sa = X_test.index[mask_sa]
    if len(idx_sa) > 0:
        if 'sensor_anomalia' in df_trabajo.columns:
            sensor_stuck = df_trabajo.loc[idx_sa, 'sensor_anomalia'].fillna('sin_sensor')
            fuente_sa = 'sensor_anomalia'
        else:
            cols_consec = [f'{c}_consecutive_unchanged' for c in COLUMNAS_SENSORES
                           if f'{c}_consecutive_unchanged' in df_trabajo.columns]
            if cols_consec:
                df_sa_hist = df_trabajo.loc[idx_sa, cols_consec].copy()
                sensor_stuck = df_sa_hist.idxmax(axis=1).str.replace('_consecutive_unchanged', '')
                fuente_sa = 'idxmax(consecutive_unchanged) fallback'
            else:
                sensor_stuck = pd.Series('sin_sensor', index=idx_sa)
                fuente_sa = 'no disponible'

        detectado_sa = pred_final.loc[idx_sa].values == 'anomalia'
        df_sa_hist = pd.DataFrame({
            'sensor_stuck': sensor_stuck.values,
            'detectado': detectado_sa,
        }, index=idx_sa)
        if 'anomalia_id' in df_trabajo.columns:
            df_sa_hist['anomalia_id'] = df_trabajo.loc[idx_sa, 'anomalia_id'].values
            df_sa_hist['sensor_stuck_detalle'] = df_sa_hist['sensor_stuck']
            anomalia_str_hist = df_sa_hist['anomalia_id'].fillna('').astype(str)
            mask_ptext_hist = df_sa_hist['sensor_stuck'].eq('PTEXT')
            mask_ptext_direct_hist = mask_ptext_hist & anomalia_str_hist.str.startswith('DIRECT_PTEXT')
            df_sa_hist.loc[mask_ptext_direct_hist, 'sensor_stuck_detalle'] = 'PTEXT_direct_short'
            df_sa_hist.loc[mask_ptext_hist & ~mask_ptext_direct_hist, 'sensor_stuck_detalle'] = 'PTEXT_stuck_standard'

        resumen_sa = df_sa_hist.groupby('sensor_stuck').agg(total=('detectado', 'count'), detectados=('detectado', 'sum'))
        resumen_sa['FN'] = resumen_sa['total'] - resumen_sa['detectados']
        resumen_sa['recall_%'] = resumen_sa['detectados'] / resumen_sa['total'] * 100
        resumen_sa = resumen_sa.sort_values('recall_%')
        lineas_sa.append(f"Fuente sensor SA: {fuente_sa}")
        for sensor, row in resumen_sa.iterrows():
            lineas_sa.append(
                f"{sensor}: total={int(row['total'])}, detectados={int(row['detectados'])}, "
                f"FN={int(row['FN'])}, recall={row['recall_%']:.1f}%"
            )
        lineas_sa.append(
            f"Total SA: {len(idx_sa)}, detectados={int(resumen_sa['detectados'].sum())}, "
            f"FN={int(resumen_sa['FN'].sum())}"
        )

        if 'sensor_stuck_detalle' in df_sa_hist.columns and df_sa_hist['sensor_stuck'].eq('PTEXT').any():
            resumen_ptext_hist = (
                df_sa_hist[df_sa_hist['sensor_stuck'].eq('PTEXT')]
                .groupby('sensor_stuck_detalle')
                .agg(total=('detectado', 'count'), detectados=('detectado', 'sum'))
            )
            resumen_ptext_hist['FN'] = resumen_ptext_hist['total'] - resumen_ptext_hist['detectados']
            resumen_ptext_hist['recall_%'] = resumen_ptext_hist['detectados'] / resumen_ptext_hist['total'] * 100
            resumen_ptext_hist = resumen_ptext_hist.sort_values('recall_%')
            lineas_ptext_sa.append('PTEXT SA por tipo de caso (filas):')
            for caso, row in resumen_ptext_hist.iterrows():
                lineas_ptext_sa.append(
                    f"{caso}: total={int(row['total'])}, detectados={int(row['detectados'])}, "
                    f"FN={int(row['FN'])}, recall={row['recall_%']:.1f}%"
                )

            seq_ptext_hist = (
                df_sa_hist[df_sa_hist['sensor_stuck'].eq('PTEXT')]
                .groupby(['sensor_stuck_detalle', 'anomalia_id'])
                .agg(detectado_any=('detectado', 'any'), filas=('detectado', 'count'))
                .reset_index()
            )
            resumen_seq_ptext_hist = seq_ptext_hist.groupby('sensor_stuck_detalle').agg(
                secuencias=('anomalia_id', 'count'),
                sec_detectadas=('detectado_any', 'sum'),
                filas=('filas', 'sum'),
            )
            resumen_seq_ptext_hist['sec_perdidas'] = resumen_seq_ptext_hist['secuencias'] - resumen_seq_ptext_hist['sec_detectadas']
            resumen_seq_ptext_hist['recall_seq_%'] = resumen_seq_ptext_hist['sec_detectadas'] / resumen_seq_ptext_hist['secuencias'] * 100
            resumen_seq_ptext_hist = resumen_seq_ptext_hist.sort_values('recall_seq_%')
            lineas_ptext_sa.append('PTEXT SA por tipo de caso (secuencias):')
            for caso, row in resumen_seq_ptext_hist.iterrows():
                lineas_ptext_sa.append(
                    f"{caso}: secuencias={int(row['secuencias'])}, detectadas={int(row['sec_detectadas'])}, "
                    f"perdidas={int(row['sec_perdidas'])}, recall_seq={row['recall_seq_%']:.1f}%, filas={int(row['filas'])}"
                )

    ptext_dur = globals().get('STUCK_DURACION_POR_SENSOR', {}).get('PTEXT', {})
    prad_dur = globals().get('STUCK_DURACION_POR_SENSOR', {}).get('PRAD', {})
    ptext_act = globals().get('STUCK_ACTIVIDAD_PREVIA', {}).get('PTEXT', {})
    reglas_ctx_hist = globals().get('REGLAS_CONTEXTO_CRUZADO', {})
    reglas_ctx_activas = [k for k, v in reglas_ctx_hist.items() if v.get('aplicar', False)]
    umbral = globals().get('UMBRAL_DETECCION', 'no_definido')
    stuck_cfg_hist = globals().get('INYECCION', {}).get('sensor_atascado', {})
    sa_fecha_test = stuck_cfg_hist.get('fecha_inicio_test_final', 'no_definido')
    sa_temporal = stuck_cfg_hist.get('secuencias_por_sensor_temporal', {}) or {}
    sa_obj_train = sum(int(v) for v in sa_temporal.get('train', {}).values()) if sa_temporal else 0
    sa_obj_test = sum(int(v) for v in sa_temporal.get('test', {}).values()) if sa_temporal else 0

    entrada = f"""
================================================================================
EJECUCION {num_ejec:03d}
================================================================================
Fecha registro: {fecha_registro}
Notebook: 04b_modelo_deteccion_lightgbm.ipynb
Configuracion principal:
- UMBRAL_DETECCION = {umbral}
- PTEXT duracion = {ptext_dur}
- PRAD duracion = {prad_dur}
- PTEXT actividad previa = {ptext_act}
- SA fecha inicio test = {sa_fecha_test}
- SA cuotas temporales = train {sa_obj_train} / test {sa_obj_test}
- Reglas contextuales activas = {reglas_ctx_activas}

Metricas finales:
Accuracy: {acc:.2f}%
TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}
Anomalia -- Precision={prec:.2f}%  Recall={rec:.2f}%  F1={f1_hist/100:.4f}

Deteccion por tipo de anomalia:
{chr(10).join(lineas_tipo) if lineas_tipo else 'No disponible'}

Sensor Atascado por sensor:
{chr(10).join(lineas_sa) if lineas_sa else 'No disponible'}

PTEXT SA por tipo de caso:
{chr(10).join(lineas_ptext_sa) if lineas_ptext_sa else 'No disponible'}
"""

    with hist_path.open('a', encoding='utf-8') as f:
        f.write(entrada + '\n')

    print(f"Historial actualizado: {hist_path}")
    print(f"Entrada anadida: EJECUCION {num_ejec:03d}")


## 9. Guardar modelo y resultados

In [ ]:
# Guardar el modelo entrenado, el imputer y la lista de features
os.makedirs(DATA_MODELS, exist_ok=True)
joblib.dump(rf_detector, MODELO_1_LGBM_PATH)
joblib.dump(imputer,     IMPUTER_M1_LGBM_PATH)

if isinstance(X_train_imputed, pd.DataFrame):
    feature_names_out = list(X_train_imputed.columns)
else:
    try:
        feature_names_out = list(imputer.get_feature_names_out(X_train.columns))
    except Exception:
        feature_names_out = [f'f{i}' for i in range(X_train_imputed.shape[1])]
joblib.dump(feature_names_out, FEATURES_M1_LGBM_PATH)

# Guardar baselines contextuales para 08_inferencia_real.ipynb
BASELINES_CTX_PATH = os.path.join(DATA_MODELS, 'baselines_ctx_lightgbm_full_v3.joblib')
if 'stats_ctx_final' in locals() and stats_ctx_final is not None:
    joblib.dump({'stats': stats_ctx_final, 'sensores': sensores_ctx}, BASELINES_CTX_PATH)
    print(f"Baselines ctx guardados: {BASELINES_CTX_PATH}")

print(f"Modelo 1 guardado en: {MODELO_1_LGBM_PATH}")

# Aplicar ctx al dataset completo antes de predecir (usando stats del train del fold final)
if 'stats_ctx_final' in locals() and stats_ctx_final is not None:
    X_full, _ = _add_ctx_features(X_train_raw, X, sensores_ctx)
else:
    X_full = X

imp_cols = feature_names_out
X_full_imp = imputer.transform(X_full)
X_full_imp_df = pd.DataFrame(X_full_imp, columns=imp_cols, index=df_trabajo.index)

idx_anomalia_full = list(rf_detector.classes_).index('anomalia')
y_pred_proba_full = rf_detector.predict_proba(X_full_imp_df)[:, idx_anomalia_full]
y_pred_base_full  = np.where(y_pred_proba_full >= UMBRAL_DETECCION, 'anomalia', 'normal')
y_pred_final_full = pd.Series(y_pred_base_full, index=df_trabajo.index, dtype='object')

# Reglas contextuales activas (mismas que en evaluaciÃ³n)
reglas_ctx = globals().get('REGLAS_CONTEXTO_CRUZADO', {
    'XHINV': {
        'ref_std_col': 'XTINV_rstd_5m',
        'ref_std_min': 0.08,
        'consec_min':  40,
        'valor_max':   98.0,
        'aplicar':     True,
    },
    'PTEXT': {
        'ref_std_col': 'XTINV_rstd_5m',
        'ref_std_min': 0.08,
        'consec_min':  10,
        'aplicar':     False,
    },
    'PRGINT': {
        'ref_std_col':        'PRAD_rstd_5m',
        'ref_std_min':        1.0,
        'consec_min':         5,
        'excluir_valor_cero': True,
        'valor_min':          5.0,
        'aplicar':            False,
    },
})

mask_regla_sa_full = pd.Series(False, index=df_trabajo.index)
for col in STUCK_SENSORES_REGLA:
    col_unchanged = f'{col}_consecutive_unchanged'
    if col_unchanged not in df_trabajo.columns:
        continue
    consecutive = df_trabajo[col_unchanged]
    mask_sensor = consecutive >= STUCK_CONSECUTIVOS_MIN
    if col in STUCK_EXCLUIR_VALOR_CERO:
        mask_sensor = mask_sensor & (df_trabajo[col] > 0.5)
    mask_regla_sa_full |= mask_sensor
y_pred_final_full.loc[mask_regla_sa_full] = 'anomalia'

ctx_forzadas_full = 0
for sensor, cfg in reglas_ctx.items():
    if not cfg.get('aplicar', False):
        continue
    col_consec  = f"{sensor}_consecutive_unchanged"
    col_ref_std = cfg['ref_std_col']
    if col_consec not in df_trabajo.columns or col_ref_std not in df_trabajo.columns:
        continue
    mask_s = (df_trabajo[col_consec] >= cfg['consec_min']) & (df_trabajo[col_ref_std] >= cfg['ref_std_min'])
    if cfg.get('excluir_valor_cero') and sensor in df_trabajo.columns:
        mask_s = mask_s & (df_trabajo[sensor] >= cfg.get('valor_min', 1.0))
    if 'valor_max' in cfg and sensor in df_trabajo.columns:
        mask_s = mask_s & (df_trabajo[sensor] < cfg['valor_max'])
    antes = (y_pred_final_full == 'anomalia').sum()
    y_pred_final_full.loc[mask_s] = 'anomalia'
    ctx_forzadas_full += int((y_pred_final_full == 'anomalia').sum() - antes)

mask_oor_full = pd.Series(False, index=df_trabajo.index)
for sensor, rango in RANGOS_FISICOS.items():
    if sensor not in df_trabajo.columns:
        continue
    valores = df_trabajo[sensor]
    mask_oor_full |= ((valores < rango['min']) | (valores > rango['max'])) & valores.notna()
y_pred_final_full.loc[mask_oor_full] = 'anomalia'

df_trabajo['pred_deteccion_base'] = y_pred_base_full
df_trabajo['pred_deteccion'] = y_pred_final_full.values
df_trabajo['pred_proba_anomalia'] = y_pred_proba_full

print(f"Pred base a umbral {UMBRAL_DETECCION}: {(df_trabajo['pred_deteccion_base'] == 'anomalia').sum():,}")
print(f"Pred final tras reglas: {(df_trabajo['pred_deteccion'] == 'anomalia').sum():,}")
print(f"Regla SA full disparos: {mask_regla_sa_full.sum():,}")
print(f"Reglas ctx activas nuevas predicciones: {ctx_forzadas_full:,}")
print(f"Regla OOR full disparos: {mask_oor_full.sum():,}")

df_trabajo.to_parquet(PARQUET_04_LGBM, index=False)
print(f"Predicciones guardadas: {PARQUET_04_LGBM}")
print(f"Shape: {df_trabajo.shape}")


In [ ]:
# Diagnostico PRAD - Sensor Atascado real
idx = X_test.index
pred = pd.Series(y_pred_detector, index=idx)
tipos = df_trabajo.loc[idx, 'etiqueta_tipo_anomalia']

df_sa = df_trabajo.loc[idx[tipos == 'Sensor Atascado']].copy()
df_sa['pred'] = pred.loc[df_sa.index]
if 'sensor_anomalia' in df_sa.columns:
    df_sa['sensor_stuck'] = df_sa['sensor_anomalia'].fillna('sin_sensor')
    print('Fuente sensor PRAD: sensor_anomalia')
else:
    cols_consec = [
        f'{col}_consecutive_unchanged'
        for col in COLUMNAS_SENSORES
        if f'{col}_consecutive_unchanged' in df_trabajo.columns
    ]
    df_sa['sensor_stuck'] = (
        df_sa[cols_consec]
        .idxmax(axis=1)
        .str.replace('_consecutive_unchanged', '', regex=False)
    )
    print('Fuente sensor PRAD: idxmax(consecutive_unchanged) [fallback]')

df_prad = df_sa[df_sa['sensor_stuck'] == 'PRAD'].copy()
df_prad['detectado'] = df_prad['pred'].eq('anomalia')

print("PRAD Sensor Atascado real")
print(f"Total: {len(df_prad)}")
if len(df_prad) == 0:
    print('No hay PRAD real en el test actual; no se puede calcular recall de PRAD en este fold.')
else:
    df_prad['Fecha'] = pd.to_datetime(df_prad['Fecha'])
    df_prad['dia'] = df_prad['Fecha'].dt.date
    df_prad['mes'] = df_prad['Fecha'].dt.to_period('M')
    df_prad['hora'] = df_prad['Fecha'].dt.hour

    print(f"Detectados TP: {df_prad['detectado'].sum()}")
    print(f"FN: {(~df_prad['detectado']).sum()}")
    print(f"Recall: {df_prad['detectado'].mean()*100:.1f}%")

    cols_diag = [
        'Fecha', 'PRAD', 'PRGINT', 'anomalia_id',
        'PRAD_consecutive_unchanged',
        'PRAD_rstd_5m',
        'PRAD_n_zero_diff_10m',
        'PRAD_stuck_vs_neighbors',
        'PRAD_pre_vs_current_rstd',
        'PRAD_rstd_vs_expected',
        'pred_proba_anomalia' if 'pred_proba_anomalia' in df_prad.columns else None,
    ]
    cols_diag = [c for c in cols_diag if c and c in df_prad.columns]

    print("\nPor mes:")
    print(pd.crosstab(df_prad['mes'], df_prad['detectado']))

    print("\nPor dia:")
    print(pd.crosstab(df_prad['dia'], df_prad['detectado']))

    print("\nPor hora:")
    print(pd.crosstab(df_prad['hora'], df_prad['detectado']))

    print("\nResumen numerico TP vs FN:")
    print(
        df_prad.groupby('detectado')[
            [c for c in [
                'PRAD',
                'PRGINT',
                'PRAD_consecutive_unchanged',
                'PRAD_rstd_5m',
                'PRAD_n_zero_diff_10m',
                'PRAD_stuck_vs_neighbors',
                'PRAD_pre_vs_current_rstd',
                'PRAD_rstd_vs_expected',
            ] if c in df_prad.columns]
        ].describe().T
    )

    print("\nFN PRAD - primeros 80:")
    display(df_prad.loc[~df_prad['detectado'], cols_diag].head(80))

    print("\nTP PRAD - primeros 80:")
    display(df_prad.loc[df_prad['detectado'], cols_diag].head(80))


In [ ]:
# Detalle FN PRAD con todas las rachas y probabilidad
if 'df_prad' not in locals() or df_prad.empty:
    print('Sin PRAD real en test; se omite detalle FN PRAD.')
else:
    cols_consec_all = [
        f'{col}_consecutive_unchanged'
        for col in COLUMNAS_SENSORES
        if f'{col}_consecutive_unchanged' in df_prad.columns
    ]

    cols_extra = [
        'Fecha', 'Hora', 'Mes',
        'PRAD', 'PRGINT',
        'PRAD_rstd_5m',
        'PRAD_n_zero_diff_10m',
        'PRAD_stuck_vs_neighbors',
        'PRAD_pre_vs_current_rstd',
        'pred',
    ]

    if 'pred_proba_anomalia' in df_prad.columns:
        cols_extra.append('pred_proba_anomalia')

    cols_extra = [c for c in cols_extra if c in df_prad.columns]
    df_fn_prad = df_prad.loc[~df_prad['detectado'], cols_extra + cols_consec_all].copy()

    if df_fn_prad.empty:
        print('PRAD real no tiene FN en test.')
    elif not cols_consec_all:
        print('No hay columnas consecutive_unchanged para detallar FN PRAD.')
    else:
        # Sensor dominante y segundo dominante: auxiliar, no ground truth.
        consec_vals = df_fn_prad[cols_consec_all]
        df_fn_prad['sensor_dom_1'] = consec_vals.idxmax(axis=1).str.replace('_consecutive_unchanged', '', regex=False)
        df_fn_prad['consec_dom_1'] = consec_vals.max(axis=1)

        second_vals = consec_vals.mask(consec_vals.eq(consec_vals.max(axis=1), axis=0))
        mask_second = second_vals.notna().any(axis=1)
        sensor_dom_2 = pd.Series('sin_segundo_sensor', index=second_vals.index, dtype='object')
        if mask_second.any():
            sensor_dom_2.loc[mask_second] = (
                second_vals.loc[mask_second]
                .idxmax(axis=1)
                .str.replace('_consecutive_unchanged', '', regex=False)
            )
        df_fn_prad['sensor_dom_2'] = sensor_dom_2
        df_fn_prad['consec_dom_2'] = second_vals.max(axis=1)

        print('FN PRAD por sensor dominante recalculado (auxiliar):')
        print(df_fn_prad['sensor_dom_1'].value_counts())

        print('\nDistribucion de probabilidad si existe:')
        if 'pred_proba_anomalia' in df_fn_prad.columns:
            print(df_fn_prad['pred_proba_anomalia'].describe())
            print('\nFN cerca del umbral 0.80:')
            display(df_fn_prad.sort_values('pred_proba_anomalia', ascending=False).head(40))

        print('\nFN PRAD con mas consecutive:')
        display(df_fn_prad.sort_values('consec_dom_1', ascending=False).head(80))

        print('\nFN PRAD con PRAD mas alto:')
        display(df_fn_prad.sort_values('PRAD', ascending=False).head(80))


## Limitaciones de PRAD SA â€” Techo fÃ­sico confirmado

**Recall PRAD SA: ~21.7% (exec 036/040). Este es el techo prÃ¡ctico con el enfoque actual.**

### Por quÃ© no mejora con mÃ¡s datos de entrenamiento

PRAD (radiaciÃ³n solar exterior) tiene una propiedad fÃ­sica que hace que los Sensor Atascado
sintÃ©ticos sean **indistinguibles** de comportamiento normal:

- **SA sintÃ©tico**: PRAD se inyecta atascado entre 14-30 muestras (7-15 min) en horas de actividad
  solar (PRAD â‰¥ 150 W/mÂ², PRGINT â‰¥ 60 W/mÂ²).
- **PRAD natural**: en dÃ­as soleados con cielo despejado, PRAD se estabiliza durante 10-30 minutos
  en el plateau del mediodÃ­a solar. Esta estabilizaciÃ³n es perfectamente real y no es una anomalÃ­a.

El modelo aprende ambos patrones y, racionalmente, los clasifica como "normal" porque el 95%
de las veces que ve esa secuencia, lo es.

**Confirmado con exec 038**: aumentar train de 10 â†’ 30 secuencias empeorÃ³ el recall a 1.8%.
El modelo se volviÃ³ ultra-conservador porque vio mÃ¡s ejemplos de "PRAD estabilizado = anomalÃ­a"
pero el espacio de features es el mismo que "PRAD estabilizado = normal".

### Caminos posibles para mejorar (requieren investigaciÃ³n)

1. **Ventanas de features mÃ¡s largas (60-120 min)**: capturar el contexto antes/despuÃ©s del plateau.
   Un SA real no tiene transiciÃ³n gradual hacia ni desde el plateau; el natural sÃ­.
2. **RestricciÃ³n horaria de inyecciÃ³n**: inyectar PRAD SA solo en horas donde no deberÃ­a haber
   plateau natural (p.ej., amanecer/atardecer, dÃ­as nublados identificados por PRGINT bajo).
3. **Feature de duraciÃ³n del plateau**: calcular cuÃ¡nto tiempo lleva PRAD sin cambiar y si eso
   es estadÃ­sticamente inusual para esa hora del dÃ­a / estaciÃ³n.
4. **Aceptar el 21.7%**: para un sistema de alertas, PRAD SA es el tipo mÃ¡s benigno (solo el
   sensor de radiaciÃ³n exterior se atasca; no implica riesgo inmediato para el cultivo).

## 10. AnÃ¡lisis estadÃ­stico de errores y umbrales

Estudio sistemÃ¡tico para decidir si vale la pena invertir esfuerzo en mejorar
**PTEXT SA (65%)** y **XTINV SA (72.1%)**.

Preguntas clave:
1. Â¿QuÃ© probabilidad asigna el modelo a los FN de PTEXT y XTINV SA?
   - Si proba â‰ˆ 0.70-0.79 â†’ bajar umbral los recuperarÃ­a con poco coste
   - Si proba < 0.50 â†’ el modelo estÃ¡ seguro de que son "normal" â†’ threshold no sirve
2. Â¿CuÃ¡l es el perfil temporal de los FP? Â¿Son evitables?
3. Â¿CuÃ¡l es el umbral Ã³ptimo para F1 global? Â¿Difiere mucho del 0.80 actual?

In [ ]:
# â”€â”€ SecciÃ³n 10: AnÃ¡lisis estadÃ­stico de errores y umbrales â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Lee directamente del parquet para no depender del estado del kernel.
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from config import TSCV_N_SPLITS, UMBRAL_DETECCION

# Usar la ruta full_v3 definida al inicio del notebook, sin reimportarla desde config.
df_full = pd.read_parquet(PARQUET_04_LGBM)
splits = list(TimeSeriesSplit(n_splits=TSCV_N_SPLITS).split(range(len(df_full))))
_, test_idx = splits[-1]
ev = df_full.iloc[test_idx].copy()

# Columnas requeridas
needed = {'etiqueta_deteccion', 'etiqueta_tipo_anomalia', 'sensor_anomalia',
          'pred_deteccion', 'pred_proba_anomalia'}
missing_cols = needed - set(ev.columns)
if missing_cols:
    print(f'Faltan columnas en parquet: {missing_cols}. Ejecuta el pipeline completo.')
else:
    ev['y_true'] = (ev['etiqueta_deteccion'] == 'anomalia').astype(int)
    ev['y_pred'] = (ev['pred_deteccion'] == 'anomalia').astype(int)
    ev['proba']  = ev['pred_proba_anomalia']

    ev['resultado'] = 'TN'
    ev.loc[(ev['y_true']==0) & (ev['y_pred']==1), 'resultado'] = 'FP'
    ev.loc[(ev['y_true']==1) & (ev['y_pred']==0), 'resultado'] = 'FN'
    ev.loc[(ev['y_true']==1) & (ev['y_pred']==1), 'resultado'] = 'TP'

    # â”€â”€ 1. DistribuciÃ³n de probabilidad por resultado â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('=== 1. Probabilidad del modelo por resultado ===')
    print(ev.groupby('resultado')['proba'].describe().round(3).to_string())

    # â”€â”€ 2. FN por tipo: probabilidad asignada â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n=== 2. FN: probabilidad por tipo de anomalÃ­a ===')
    fn = ev[ev['resultado'] == 'FN'].copy()
    def proba_stats(g):
        return pd.Series({
            'FN': len(g),
            'media': round(g['proba'].mean(), 3),
            'mediana': round(g['proba'].median(), 3),
            'pct>=0.60': round((g['proba'] >= 0.60).mean(), 3),
            'pct>=0.70': round((g['proba'] >= 0.70).mean(), 3),
            'pct>=0.75': round((g['proba'] >= 0.75).mean(), 3),
        })
    print(fn.groupby('etiqueta_tipo_anomalia').apply(proba_stats).to_string())

    # â”€â”€ 3. FN SA por sensor â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n=== 3. FN Sensor Atascado: probabilidad por sensor ===')
    fn_sa = fn[fn['etiqueta_tipo_anomalia'] == 'Sensor Atascado']
    if 'sensor_anomalia' in fn_sa.columns and not fn_sa.empty:
        print(fn_sa.groupby('sensor_anomalia').apply(proba_stats).to_string())

    # â”€â”€ 4. FP por hora â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n=== 4. FP: distribuciÃ³n por hora del dÃ­a ===')
    fp = ev[ev['resultado'] == 'FP']
    if 'Hora' in fp.columns:
        fp_hora = fp.groupby('Hora').size()
        print(fp_hora[fp_hora > 0].to_string())
    print(f'Total FP={len(fp)} | proba media FP={fp["proba"].mean():.3f} | '
          f'proba mediana FP={fp["proba"].median():.3f}')

    # â”€â”€ 5. Curva de umbrales â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n=== 5. Curva de umbrales (solo RF, sin post-procesado) ===')
    y_true = ev['y_true'].values
    probas  = ev['proba'].values
    rows = []
    for thr in np.arange(0.30, 0.96, 0.05):
        yp = (probas >= thr).astype(int)
        tp = int(((y_true==1) & (yp==1)).sum())
        fp_n = int(((y_true==0) & (yp==1)).sum())
        fn_n = int(((y_true==1) & (yp==0)).sum())
        prec = tp/(tp+fp_n) if (tp+fp_n)>0 else 0.0
        rec  = tp/(tp+fn_n) if (tp+fn_n)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        rows.append({'thr':round(thr,2),'TP':tp,'FP':fp_n,'FN':fn_n,
                     'Precision':round(prec,4),'Recall':round(rec,4),'F1':round(f1,4)})
    df_thr = pd.DataFrame(rows)
    print(df_thr.to_string(index=False))
    opt = df_thr.loc[df_thr['F1'].idxmax()]
    print(f'\nUmbral optimo F1: {opt["thr"]} -> F1={opt["F1"]:.4f} | FP={opt["FP"]} FN={opt["FN"]}')

    # â”€â”€ 6. QuÃ© umbral recupera FN de PTEXT y XTINV SA â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    print('\n=== 6. Coste-beneficio de bajar umbral para PTEXT y XTINV SA ===')
    for sensor in ['PTEXT', 'XTINV', 'PRAD']:
        mask_sa = ((ev['etiqueta_tipo_anomalia'] == 'Sensor Atascado') &
                   (ev['sensor_anomalia'] == sensor))
        df_s = ev[mask_sa]
        total = len(df_s)
        if total == 0:
            continue
        print(f'\n{sensor} SA (total={total} filas en test fold):')
        for thr in [0.80, 0.75, 0.70, 0.65, 0.60, 0.50]:
            det = int((df_s['proba'] >= thr).sum())
            fp_global = int(((ev['y_true']==0) & (ev['proba']>=thr)).sum())
            fn_global  = int(((ev['y_true']==1) & (ev['proba']< thr)).sum())
            print(f'  thr={thr:.2f}: {sensor}_det={det}/{total} ({100*det/total:.1f}%) | '
                  f'FP_global={fp_global} | FN_global={fn_global}')